**CART TREE**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

class CARTTree():

    def __init__(self, max_depth=None):
        self.max_depth = max_depth

    @staticmethod
    def _purity(y):

        classes=y.unique()

        return len(classes)==1

    def _set_df_type(self, X, dtype):
        X = X.astype(dtype)
        return X

    def _stopping_conditions(self,y, depth):
        if len(y) == 0:
            return True, True
        return (self._purity(y), depth == self.max_depth and self.max_depth is not None)

    @staticmethod
    def _gini_impurity(y):
        _, counts_classes = np.unique(y, return_counts=True)  #k
        squared_probabilities = np.square(counts_classes / y.size)
        gini_impurity = 1 - sum(squared_probabilities)

        return gini_impurity

    def return_leaf(self,y):
        return y.mode()[0]

    @staticmethod
    def _cost_function(left_df, right_df, method):
            total_df_size = left_df.size + right_df.size
            p_left_df = left_df.size / total_df_size
            p_right_df = right_df.size / total_df_size
            J_left = method(left_df)
            J_right = method(right_df)
            J = p_left_df*J_left + p_right_df*J_right
            return J

    def _grow_tree(self,x,y, depth=0):

        if len(y) == 0 or len(x) == 0:
            leaf_node = f'leaf: {self.return_leaf(y)}'
            return leaf_node

        if any(self._stopping_conditions(y, depth)):
            leaf_node = f'leaf: {self.return_leaf(y)}'
            return leaf_node

        features = x.columns
        min_cf = np.inf
        best_feature, best_threshold = None, None

        for feature in features:

            column = x.loc[:, feature]
            unique_feature_values = column.unique()

            if len(unique_feature_values) <= 1:
                continue

            for i in range(1,len(unique_feature_values)):
                cur_value = unique_feature_values[i]
                prev_value = unique_feature_values[i-1]
                av_value = (cur_value+prev_value) /2

                left_x = x[x[feature]>=av_value].index
                right_x = x[~(x[feature]>=av_value)].index

                left_y = y.loc[left_x]
                right_y = y.loc[right_x]

                cf_res = self._cost_function(left_y,right_y, self._gini_impurity)

                if cf_res < min_cf:
                    min_cf = cf_res
                    best_feature, best_threshold = feature,av_value

        #print(min_cf,best_feature, best_threshold)

        if best_feature is None:
            return f'leaf: {self.return_leaf(y)}'

        right_indexes = x[x[best_feature]>=best_threshold].index
        left_indexes = x[x[best_feature]<best_threshold].index
        left_node = x.loc[left_indexes]
        right_node = x.loc[right_indexes]
        left_labels, right_labels = y.loc[left_indexes], y.loc[right_indexes]

        decision_node = f'{best_feature} <= {best_threshold}'
        tree = {decision_node: []}
        left_subtree = self._grow_tree(left_node, left_labels,depth+1)
        right_subtree = self._grow_tree(right_node, right_labels,depth+1)

        if left_subtree == right_subtree:
            tree = left_subtree
        else:
            tree[decision_node].extend([left_subtree, right_subtree])

        return tree

    def fit(self, x, y):

        self.tree_ = self._grow_tree(x, y)

    def predict(self, x):
        if hasattr(self, 'tree_'):
            return x.apply(self._predict_row, axis=1, tree=self.tree_)
        else:
            raise Exception("Дерево не обучено")

    def _predict_row(self, row, tree):

        if isinstance(tree, str) and tree.startswith('leaf'):
            return tree.split(':')[1].strip()

        node = list(tree.keys())[0]
        feature, condition = node.split(' <= ')
        threshold = float(condition)

        if row[feature] <= threshold:
            subtree = tree[node][0]
        else:
            subtree = tree[node][1]

        return self._predict_row(row, subtree)


**C4.5 TREE**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import copy

class C45Tree():

    def __init__(self, max_depth=None, min_samples_leaf=None, ccp_alpha=0.0):
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.ccp_alpha = ccp_alpha  # параметр регуляризации для пост-прунинга
        self.tree_ = None
        self.node_count_ = 0  # общее колво узлов

    @staticmethod
    def _purity(y):
        classes = y.unique()
        return len(classes) == 1

    def _set_df_type(self, X, dtype):
        X = X.astype(dtype)
        return X

    def _stopping_conditions(self, y, depth):
        if len(y) == 0:
            return True, True

        return (self._purity(y),
                (self.max_depth is not None and depth >= self.max_depth) or
                (self.min_samples_leaf is not None and len(y) <= self.min_samples_leaf))

    def return_leaf(self, y):
        if len(y) == 0:
            return None
        return y.mode()[0]

    @staticmethod
    def _entropy(labels_y):
        if len(labels_y) == 0:
            return 0.0

        classes_labels = np.unique(labels_y)
        info_s = 0
        n_total = len(labels_y)

        for clas in classes_labels:
            mask = (labels_y == clas)
            count = np.sum(mask)
            proportion = count / n_total
            if proportion > 0:
                info_s -= proportion * np.log(proportion)
        return info_s

    @staticmethod
    def _entropy_by_A(values, labels, method):
        n_total = len(values)
        if n_total == 0:
            return 0.0

        info_a_s = 0
        uniq_values = np.unique(values.dropna())

        for value in uniq_values:
            mask = values == value
            proportion = np.sum(mask) / n_total
            if proportion > 0:
                branch_labels = labels[mask]
                branch_entropy = method(branch_labels)
                info_a_s += proportion * branch_entropy
        return info_a_s

    @staticmethod
    def _inf_gain(info_s, info_a_s):
        return info_s - info_a_s

    @staticmethod
    def _split_info(values_x):
        n_total = len(values_x)
        if n_total == 0:
            return 0

        split_info = 0
        uniq_values = np.unique(values_x.dropna())

        for value in uniq_values:
            mask = values_x == value
            count = np.sum(mask)
            proportion = count / n_total
            if proportion > 0:
                split_info -= proportion * np.log(proportion)
        return split_info

    @staticmethod
    def _gain_ratio(inf_gain, split_info):
        if split_info == 0:
            return 0
        return max(0.0, inf_gain) / split_info

    @staticmethod
    def _is_numeric(series):
        try:
            pd.to_numeric(series.dropna())
            return True
        except (ValueError, TypeError):
            return False

    @staticmethod
    def _entropy_by_A_numeric(values, labels, method):

        df = pd.DataFrame({'value': values, 'label': labels}).dropna()
        df = df.sort_values('value')

        if len(df) == 0:
            return 0.0, None

        unique_values = df['value'].unique()
        if len(unique_values) == 1:
            return 0.0, None

        best_entropy = float('inf')
        best_threshold = None
        n_total = len(df)

        for i in range(len(unique_values) - 1):
            threshold = (unique_values[i] + unique_values[i + 1]) / 2

            left_mask = df['value'] <= threshold
            right_mask = df['value'] > threshold

            left_labels = df.loc[left_mask, 'label']
            right_labels = df.loc[right_mask, 'label']

            if len(left_labels) == 0 or len(right_labels) == 0:
                continue

            left_entropy = method(left_labels)
            right_entropy = method(right_labels)

            weighted_entropy = (len(left_labels) / n_total) * left_entropy + (len(right_labels) / n_total) * right_entropy

            if weighted_entropy < best_entropy:
                best_entropy = weighted_entropy
                best_threshold = threshold

        return best_entropy, best_threshold

    @staticmethod
    def _split_info_numeric(values_x, threshold):
        n_total = len(values_x)
        if n_total == 0:
            return 0

        left_count = np.sum(values_x <= threshold)
        right_count = np.sum(values_x > threshold)

        split_info = 0
        for count in [left_count, right_count]:
            if count > 0:
                p = count / n_total
                split_info -= p * np.log(p)

        return split_info

    def evaluate_all_features(self, x, y, available_features, verbose=False):
        results = {}
        thresholds = {}

        for feature in available_features:
            values = x[feature]
            labels = y

            info_s = self._entropy(labels)

            if self._is_numeric(values):
                info_a_s, threshold = self._entropy_by_A_numeric(values, labels, self._entropy)
                if threshold is None:
                    gain_ratio = 0
                else:
                    inf_gain = self._inf_gain(info_s, info_a_s)
                    split_info = self._split_info_numeric(values, threshold)
                    gain_ratio = self._gain_ratio(inf_gain, split_info)
            else:
                info_a_s = self._entropy_by_A(values, labels, self._entropy)
                inf_gain = self._inf_gain(info_s, info_a_s)
                split_info = self._split_info(values)
                gain_ratio = self._gain_ratio(inf_gain, split_info)
                threshold = None

            results[feature] = gain_ratio
            thresholds[feature] = threshold

            if verbose:
                print(f"Признак: {feature}")
                print(f"  GainRatio = {gain_ratio:.4f}\n")

        return results, thresholds

    def _split_data_by_feature(self, x, y, feature, threshold=None):
        branches = {}

        if threshold is not None:
            mask_left = x[feature] <= threshold
            mask_right = x[feature] > threshold

            branches[f'≤{threshold:.4f}'] = (x.loc[mask_left], y.loc[mask_left])
            branches[f'>{threshold:.4f}'] = (x.loc[mask_right], y.loc[mask_right])
        else:
            unique_values = x[feature].dropna().unique()
            for value in unique_values:
                mask = x[feature] == value
                branches[value] = (x.loc[mask], y.loc[mask])

        return branches

    def _grow_tree(self, x, y, available_features, depth=0):

        if len(y) == 0 or len(x) == 0:
            return {
                'type': 'leaf',
                'value': self.return_leaf(y),
                'n_samples': len(y)
            }

        is_pure, stop_condition = self._stopping_conditions(y, depth)

        if is_pure or stop_condition:
            return {
                'type': 'leaf',
                'value': self.return_leaf(y),
                'n_samples': len(y)
            }

        gain_ratios, thresholds = self.evaluate_all_features(x, y, available_features, verbose=False)

        if not gain_ratios:
            return {
                'type': 'leaf',
                'value': self.return_leaf(y),
                'n_samples': len(y)
            }

        best_feature = max(gain_ratios, key=gain_ratios.get)
        best_gain_ratio = gain_ratios[best_feature]

        if best_gain_ratio <= 0:
            return {
                'type': 'leaf',
                'value': self.return_leaf(y),
                'n_samples': len(y)
            }

        threshold = thresholds[best_feature]

        branches = self._split_data_by_feature(x, y, best_feature, threshold)

        for branch_x, branch_y in branches.values():
            if self.min_samples_leaf is not None:
                if len(branch_y) < self.min_samples_leaf:
                    return {
                        'type': 'leaf',
                        'value': self.return_leaf(y),
                        'n_samples': len(y)
                    }

        node = {
            'type': 'node',
            'feature': best_feature,
            'threshold': threshold,
            'children': {},
            'n_samples': len(y),
            'class_distribution': y.value_counts().to_dict()
        }

        for branch_value, (branch_x, branch_y) in branches.items():
            if len(branch_y) == 0:
                node['children'][branch_value] = {
                    'type': 'leaf',
                    'value': self.return_leaf(y),
                    'n_samples': 0
                }
            else:
                remaining_features = [f for f in available_features if f != best_feature]
                node['children'][branch_value] = self._grow_tree(
                    branch_x, branch_y, remaining_features, depth + 1
                )

        return node

    def _calculate_impurity(self, node):
        if node['type'] == 'leaf':
            return 0.0

        total_samples = node['n_samples']
        if total_samples == 0:
            return 0.0

        weighted_impurity = 0.0
        for child in node['children'].values():
            if child['n_samples'] > 0:
                if child['type'] == 'leaf':
                    child_impurity = 0.0  # Лист чистый
                else:
                    child_impurity = self._calculate_tree_impurity(child)

                weighted_impurity += (child['n_samples'] / total_samples) * child_impurity

        return weighted_impurity

    def _calculate_tree_impurity(self, node):
        if node['type'] == 'leaf':
            return 0.0

        total_impurity = 0.0
        for child in node['children'].values():
            total_impurity += self._calculate_tree_impurity(child)

        return total_impurity

    def _count_leaves(self, node):
        if node['type'] == 'leaf':
            return 1

        leaf_count = 0
        for child in node['children'].values():
            leaf_count += self._count_leaves(child)

        return leaf_count

    def _compute_ccp_alpha(self, node, parent_impurity=None):

        if node['type'] == 'leaf':
            return []

        alphas = []

        total_samples = node['n_samples']
        if total_samples > 0:
            class_dist = node.get('class_distribution', {})
            R_t = 0.0
            for count in class_dist.values():
                p = count / total_samples
                if p > 0:
                    R_t -= p * np.log(p)
        else:
            R_t = 0.0

        R_Tt = self._calculate_tree_impurity(node)

        leaf_count = self._count_leaves(node)

        if leaf_count > 1:
            alpha = (R_t - R_Tt) / (leaf_count - 1)
            if alpha >= 0:
                alphas.append((alpha, node))

        for child in node['children'].values():
            alphas.extend(self._compute_ccp_alpha(child))

        return alphas

    def _prune_tree(self, node, alpha_threshold):

        if node['type'] == 'leaf':
            return node

        new_children = {}
        for branch_key, child in node['children'].items():
            new_children[branch_key] = self._prune_tree(child, alpha_threshold)
        node['children'] = new_children

        total_samples = node['n_samples']
        if total_samples == 0:
            node['type'] = 'leaf'
            node['value'] = None
            return node

        R_t = 0.0
        class_dist = node.get('class_distribution', {})
        for count in class_dist.values():
            p = count / total_samples
            if p > 0:
                R_t -= p * np.log(p)

        R_Tt = self._calculate_tree_impurity(node)
        leaf_count = self._count_leaves(node)

        if leaf_count > 1:
            alpha = (R_t - R_Tt) / (leaf_count - 1)

            if alpha <= alpha_threshold:
                leaf_value = max(class_dist, key=class_dist.get) if class_dist else None
                return {
                    'type': 'leaf',
                    'value': leaf_value,
                    'n_samples': total_samples
                }

        return node

    def fit(self, x, y):
        available_features = x.columns
        self.tree_ = self._grow_tree(x, y, available_features)

        if self.ccp_alpha > 0 and self.tree_ is not None:
            self.tree_ = self._prune_tree(self.tree_, self.ccp_alpha)

        return self

    def _predict_row(self, row, tree):

        if tree['type'] == 'leaf':
            return tree['value']

        feature = tree['feature']
        threshold = tree['threshold']

        if feature not in row.index:
            return None

        value = row[feature]

        if pd.isna(value):
            class_dist = tree.get('class_distribution', {})
            return max(class_dist, key=class_dist.get) if class_dist else None

        if threshold is not None:
            if value <= threshold:
                branch_key = f'≤{threshold:.4f}'
            else:
                branch_key = f'>{threshold:.4f}'
        else:
            branch_key = value

        if branch_key in tree['children']:
            return self._predict_row(row, tree['children'][branch_key])
        else:
            class_dist = tree.get('class_distribution', {})
            return max(class_dist, key=class_dist.get) if class_dist else None

    def predict(self, x):
        if not hasattr(self, 'tree_') or self.tree_ is None:
            raise Exception("Дерево не обучено")

        return x.apply(lambda row: self._predict_row(row, self.tree_), axis=1)

    def get_params(self, deep=True):
        return {
            'max_depth': self.max_depth,
            'min_samples_leaf': self.min_samples_leaf,
            'ccp_alpha': self.ccp_alpha
        }

    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self


In [ ]:
import time
from pprint import pprint
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X_clf, y_clf = make_classification(
    n_samples=200,
    n_features=5,
    n_informative=4,
    n_redundant=1,
    n_classes=3,
    n_clusters_per_class=1,
    random_state=42
)
df = pd.DataFrame(
    X_clf,
    columns=[f'feature_{i+1}' for i in range(X_clf.shape[1])]
)
df['target'] = y_clf
print(df)

X = df.drop('target', axis=1)  # или df.iloc[:, :-1]
y = df['target']                # или df.iloc[:, -1]

X1_train, X1_test, y1_train, y1_test = train_test_split(
    X, y, test_size=0.3, random_state=0
)

print("\n" + "="*60)
print("СРАВНЕНИЕ ТРЕХ АЛГОРИТМОВ НА Iris DATASET")
print("="*60)

# ============== 1. ВАШ CART (MyDecisonTree) ==============
print("\n1. ВАШ CART (MyDecisonTree)")
print("-" * 40)

start_train = time.time()
dt = CARTTree(max_depth=8)
dt.fit(X1_train, y1_train)
train_time_cart = time.time() - start_train

start_predict = time.time()
pred = dt.predict(X1_test).astype(int)
predict_time_cart = time.time() - start_predict
accuracy = accuracy_score(y1_test,pred)
print(f"  Точность: {accuracy:.4f}")
print(f"  Время обучения: {train_time_cart:.6f} сек")
print(f"  Время предсказания: {predict_time_cart:.6f} сек")


# ============== 2. Scikit-learn ==============
print("\n3. Scikit-learn DecisionTreeClassifier")
print("-" * 40)

start_train = time.time()
sk_tree = DecisionTreeClassifier(random_state=0, max_depth=6, min_samples_leaf=1)
sk_tree.fit(X1_train, y1_train)
train_time_sk = time.time() - start_train

start_predict = time.time()
pred_sk = sk_tree.predict(X1_test)
predict_time_sk = time.time() - start_predict

accuracy_sk = accuracy_score(y1_test,pred_sk)
print(f"  Точность: {accuracy_sk:.4f}")
print(f"  Время обучения: {train_time_sk:.6f} сек")
print(f"  Время предсказания: {predict_time_sk:.6f} сек")

# ============== 3. C4.5 (без прунинга) ==============
print("\n2. C4.5 (твоя реализация)")
print("-" * 40)
start_train = time.time()
tree_no_prune = C45Tree(max_depth=10, min_samples_leaf=1, ccp_alpha=0.0)
tree_no_prune.fit(X1_train, y1_train)
train_time_no_prune = time.time() - start_train

start_predict = time.time()
pred_no_prune = tree_no_prune.predict(X1_test)
predict_time_no_prune = time.time() - start_predict

accuracy_no_prune = accuracy_score(y1_test,pred_no_prune)
print(f"  Точность: {accuracy_no_prune:.4f}")
print(f"  Время обучения: {train_time_no_prune:.6f} сек")
print(f"  Время предсказания: {predict_time_no_prune:.6f} сек")

# ============== 4. C4.5 (слабое дерево, ccp_alpha=0.01) ==============
print("\n3. C4.5 (слабое дерево, ccp_alpha=0.01)")
print("-" * 40)
start_train = time.time()
tree_weak = C45Tree(max_depth=10, min_samples_leaf=1, ccp_alpha=0.01)
tree_weak.fit(X1_train, y1_train)
train_time_weak = time.time() - start_train

start_predict = time.time()
pred_weak = tree_weak.predict(X1_test)
predict_time_weak = time.time() - start_predict

accuracy_weak = accuracy_score(y1_test, pred_weak)
print(f"  Точность: {accuracy_weak:.4f}")
print(f"  Время обучения: {train_time_weak:.6f} сек")
print(f"  Время предсказания: {predict_time_weak:.6f} сек")

# ============== 5. C4.5 (сильное дерево, ccp_alpha=0.1) ==============
print("\n4. C4.5 (сильное дерево, ccp_alpha=0.1)")
print("-" * 40)
start_train = time.time()
tree_strong = C45Tree(max_depth=10, min_samples_leaf=1, ccp_alpha=0.05)
tree_strong.fit(X1_train, y1_train)
train_time_strong = time.time() - start_train

start_predict = time.time()
pred_strong = tree_strong.predict(X1_test)
predict_time_strong = time.time() - start_predict

accuracy_strong = accuracy_score(y1_test, pred_strong)
print(f"  Точность: {accuracy_strong:.4f}")
print(f"  Время обучения: {train_time_strong:.6f} сек")
print(f"  Время предсказания: {predict_time_strong:.6f} сек")


# ============== ИТОГОВАЯ ТАБЛИЦА ==============
print("\n" + "="*60)
print("ИТОГОВОЕ СРАВНЕНИЕ")
print("="*60)

print(f"{'Алгоритм':<30} {'Точность':<12} {'Время обучения':<15} {'Время предсказания':<15}")
print("-" * 72)
print(f"{'Ваш CART':<30} {accuracy:<12.4f} {train_time_cart:<15.6f} {predict_time_cart:<15.6f}")
print(f"{'Scikit-learn':<30} {accuracy_sk:<12.4f} {train_time_sk:<15.6f} {predict_time_sk:<15.6f}")
print(f"{'C4.5 (без прунинга)':<30} {accuracy_no_prune:<12.4f} {train_time_no_prune:<15.6f} {predict_time_no_prune:<15.6f}")
print(f"{'C4.5 (слабое, α=0.01)':<30} {accuracy_weak:<12.4f} {train_time_weak:<15.6f} {predict_time_weak:<15.6f}")
print(f"{'C4.5 (сильное, α=0.1)':<30} {accuracy_strong:<12.4f} {train_time_strong:<15.6f} {predict_time_strong:<15.6f}")
print("-"*72)

     feature_1  feature_2  feature_3  feature_4  feature_5  target
0    -0.802119  -0.882476  -2.693626   0.576265   2.690074       1
1     1.309142  -1.290732  -0.438623   2.187527   0.887820       2
2     1.044304  -1.004591  -0.073299  -0.933062  -0.760595       0
3    -1.766672  -0.958149  -1.374853   1.499995   2.354526       1
4     1.422870   2.168135  -1.068934   0.946404   0.944541       2
..         ...        ...        ...        ...        ...     ...
195  -2.877567  -1.356390  -0.430663  -1.924462   0.338304       0
196  -0.173054  -0.471297  -3.110327  -1.754091   1.760452       2
197  -1.330180  -2.174492  -3.807039  -1.374013   2.807923       2
198  -0.881981  -1.026275  -0.172027   0.460564   0.596844       1
199   0.547412  -1.288583  -0.635213  -0.933511  -0.152122       0

[200 rows x 6 columns]

СРАВНЕНИЕ ТРЕХ АЛГОРИТМОВ НА Iris DATASET

1. ВАШ CART (MyDecisonTree)
----------------------------------------
  Точность: 0.8333
  Время обучения: 1.521498 сек
  Время п

**Обертки над деревьями с подддержкой RSM**

In [ ]:
class RSMWrapperCARTTree(CARTTree):

  def __init__(self, max_depth=None, min_samples_leaf=None, max_features=None):
      super().__init__(max_depth=max_depth)
      self.max_depth = max_depth
      self.min_samlples_leaf = min_samples_leaf
      self.max_features = max_features

  def _grow_tree(self,x,y, depth=0):

        if len(y) == 0 or len(x) == 0:
            leaf_node = f'leaf: {self.return_leaf(y)}'
            return leaf_node

        if any(self._stopping_conditions(y, depth)):
            leaf_node = f'leaf: {self.return_leaf(y)}'
            return leaf_node

        features = x.columns.tolist()
        rsm_features = np.random.choice(features, size = self.max_features, replace=False)
        min_cf = np.inf
        best_feature, best_threshold = None, None

        for feature in rsm_features:

            column = x.loc[:, feature]
            unique_feature_values = column.unique()

            if len(unique_feature_values) <= 1:
                continue

            for i in range(1,len(unique_feature_values)):
                cur_value = unique_feature_values[i]
                prev_value = unique_feature_values[i-1]
                av_value = (cur_value+prev_value) /2

                left_x = x[x[feature]>=av_value].index
                right_x = x[~(x[feature]>=av_value)].index

                left_y = y.loc[left_x]
                right_y = y.loc[right_x]

                cf_res = self._cost_function(left_y,right_y, self._gini_impurity)

                if cf_res < min_cf:
                    min_cf = cf_res
                    best_feature, best_threshold = feature,av_value

        #print(min_cf,best_feature, best_threshold)

        if best_feature is None:
            return f'leaf: {self.return_leaf(y)}'

        right_indexes = x[x[best_feature]>=best_threshold].index
        left_indexes = x[x[best_feature]<best_threshold].index
        left_node = x.loc[left_indexes]
        right_node = x.loc[right_indexes]
        left_labels, right_labels = y.loc[left_indexes], y.loc[right_indexes]

        decision_node = f'{best_feature} <= {best_threshold}'
        tree = {decision_node: []}
        left_subtree = self._grow_tree(left_node, left_labels,depth+1)
        right_subtree = self._grow_tree(right_node, right_labels,depth+1)

        if left_subtree == right_subtree:
            tree = left_subtree
        else:
            tree[decision_node].extend([left_subtree, right_subtree])

        return tree

In [ ]:
class RSMWrapperC45Tree(C45Tree):

      def __init__(self, max_depth=None, min_samples_leaf=None, ccp_alpha=0.0, max_features = None):
        super().__init__(max_depth=max_depth, min_samples_leaf=min_samples_leaf, ccp_alpha=ccp_alpha)
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.ccp_alpha = ccp_alpha  # параметр регуляризации для пост-прунинга
        self.tree_ = None
        self.node_count_ = 0  # общее колво узлов
        self.max_features = max_features

      def _grow_tree(self, x, y, available_features, depth=0):

        if len(y) == 0 or len(x) == 0:
            return {
                'type': 'leaf',
                'value': self.return_leaf(y),
                'n_samples': len(y)
            }

        is_pure, stop_condition = self._stopping_conditions(y, depth)

        if is_pure or stop_condition:
            return {
                'type': 'leaf',
                'value': self.return_leaf(y),
                'n_samples': len(y)
            }

        self.max_features = len(available_features) if self.max_features > len(available_features) else self.max_features
        rsm_features = np.random.choice(available_features, size = self.max_features, replace=False)

        gain_ratios, thresholds = self.evaluate_all_features(x, y, rsm_features, verbose=False)

        if not gain_ratios:
            return {
                'type': 'leaf',
                'value': self.return_leaf(y),
                'n_samples': len(y)
            }

        best_feature = max(gain_ratios, key=gain_ratios.get)
        best_gain_ratio = gain_ratios[best_feature]

        if best_gain_ratio <= 0:
            return {
                'type': 'leaf',
                'value': self.return_leaf(y),
                'n_samples': len(y)
            }

        threshold = thresholds[best_feature]

        branches = self._split_data_by_feature(x, y, best_feature, threshold)

        for branch_x, branch_y in branches.values():
            if self.min_samples_leaf is not None:
                if len(branch_y) < self.min_samples_leaf:
                    return {
                        'type': 'leaf',
                        'value': self.return_leaf(y),
                        'n_samples': len(y)
                    }

        node = {
            'type': 'node',
            'feature': best_feature,
            'threshold': threshold,
            'children': {},
            'n_samples': len(y),
            'class_distribution': y.value_counts().to_dict()
        }

        for branch_value, (branch_x, branch_y) in branches.items():
            if len(branch_y) == 0:
                node['children'][branch_value] = {
                    'type': 'leaf',
                    'value': self.return_leaf(y),
                    'n_samples': 0
                }
            else:
                remaining_features = [f for f in available_features if f != best_feature]
                node['children'][branch_value] = self._grow_tree(
                    branch_x, branch_y, remaining_features, depth + 1
                )

        return node

      def fit(self, x, y):
        available_features = x.columns.tolist()

        self.tree_ = self._grow_tree(x, y, available_features)

        if self.ccp_alpha > 0 and self.tree_ is not None:
            self.tree_ = self._prune_tree(self.tree_, self.ccp_alpha)

        return self

**Random Forest**

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score
import asyncio
from tqdm import tqdm
from joblib import Parallel, delayed

class MyRandomForestClassifier():
  def __init__(self, tree_model, n_estimators=100, max_depth=10, min_samples_leaf=1, max_features='sqrt',oob_validation = True, bootstrap = True, **kwargs):
    self.tree_model = tree_model
    self.n_estimators = n_estimators
    self.max_depth = max_depth
    self.min_samples_leaf = min_samples_leaf
    self.oob_validation = oob_validation
    self.bootstrap = bootstrap
    self.features_limit = max_features
    self.other_params = kwargs
    self._lock = asyncio.Lock()


  def _max_features_mechanism(self, type, n_features):
      if isinstance(type, (int, float)):
        res = min(int(type), n_features)
        if res <= 0:
              raise ValueError(f"less than zero")
        return res
      elif type is not None:
        if type.lower() == "sqrt":
          return max(1,int(np.sqrt(n_features)))
        if type.lower() == "log2":
          return max(1,int(np.log2(n_features)))
        else:
          raise ValueError(f"unknown type")
      else:
        return n_features

  def _bootstrap_sampling(self, X,y,n_samples):
    if self.bootstrap:
      sample_indeces = np.random.choice(n_samples, size = n_samples, replace = True)
      if self.oob_validation:
        all_indeces = range(n_samples)
        oob_indeces = list(set(all_indeces) - set(sample_indeces))
        return X.iloc[sample_indeces], y.iloc[sample_indeces], X.iloc[oob_indeces], y.iloc[oob_indeces]
      return X.iloc[sample_indeces], y.iloc[sample_indeces], None, None  #.iloc
    else:
      return X, y, None, None

  def _train_single_tree(self, X, y, n_samples):

        tree, oob_score = None, None

        X_sample, y_sample, X_oob, y_oob = self._bootstrap_sampling(X, y, n_samples)
        tree = self.tree_model(max_depth = self.max_depth, min_samples_leaf = self.min_samples_leaf, max_features = self.max_features, **self.other_params)
        tree.fit(X_sample, y_sample)
        #self.ensemble.append(tree)  #каждый процесс имеет свой self.ensemble и не возвращает результат через append, поэтому нужно напрямую из процесса возвращать рещультат

        if self.oob_validation and X_oob is not None:
          y_oob_pred = tree.predict(X_oob)
          oob_score = accuracy_score(y_oob, y_oob_pred)

        return tree, oob_score


  def _parallel_train_trees_completer(self, X, y, n_jobs):

        n_samples, n_features = X.shape
        self.classes = np.unique(y)

        self.ensemble = []
        self.oob_scores = []
        self.average_oob_score = None

        self.max_features = self._max_features_mechanism(self.features_limit, n_features)

        results = Parallel(n_jobs=n_jobs)(
            delayed(self._train_single_tree)(X, y, n_samples)
            for i in tqdm(range(self.n_estimators), desc = "parallel training process")
        )

        for res in results:
          if res[0] is not None:
            self.ensemble.append(res[0])
          if res[1] is not None:
            self.oob_scores.append(res[1])

        if self.oob_validation and self.oob_scores:
          self.average_oob_score = np.mean(self.oob_scores)

        return self

  def parallel_fit(self, X, y, n_jobs = -1):
        return self._parallel_train_trees_completer(X, y, n_jobs)

  async def _tree_task(self, X, y, n_samples, lock):
      X_sample, y_sample, X_oob, y_oob = self._bootstrap_sampling(X,y, n_samples)
      tree = self.tree_model(max_depth = self.max_depth, min_samples_leaf = self.min_samples_leaf, max_features = self.max_features, **self.other_params)
      tree.fit(X_sample, y_sample)
      if lock:
        async with self._lock:
          self.ensemble.append(tree)
      else:
          self.ensemble.append(tree)

      if self.oob_validation and X_oob is not None:
          y_oob_pred = tree.predict(X_oob)
          oob_score = accuracy_score(y_oob, y_oob_pred)
          if lock:
            async with self._lock:
              self.oob_scores.append(oob_score)
          else:
              self.oob_scores.append(oob_score)

  async def _tree_tasks_completer(self, X, y, lock=True):  # в colav/jupyter запускать напрямую прост как корутину чрез await
    n_samples, n_features = X.shape
    self.classes = np.unique(y)

    self.ensemble = []
    self.oob_scores = []
    self.average_oob_score = None

    self.max_features = self._max_features_mechanism(self.features_limit, n_features)

    tasks = [self._tree_task(X, y, n_samples, lock) for i in range(self.n_estimators)]

    await asyncio.gather(*tasks)

    if self.oob_validation and self.oob_scores:
        self.average_oob_score = np.mean(self.oob_scores)

    return self

  def async_fit(self, X, y, lock=True):  #плохо работает, т.к. задача cpu bound
    try:
      loop = asyncio.get_event_loop()  #get_running_loop() вызовет ошибку
      return loop.run_until_complete(self._tree_tasks_completer(X, y, lock=lock))
    except RuntimeError:
        return asyncio.run(self._tree_tasks_completer(X, y, lock=lock))

  def fit(self, X, y): # расширить до async await gather

      n_samples, n_features = X.shape
      self.classes = np.unique(y)

      self.ensemble = []
      self.oob_scores = []
      self.average_oob_score = None

      self.max_features = self._max_features_mechanism(self.features_limit, n_features)

      for i in tqdm(range(self.n_estimators), desc="training process"):
        X_sample, y_sample, X_oob, y_oob = self._bootstrap_sampling(X,y, n_samples)
        tree = self.tree_model(max_depth = self.max_depth, min_samples_leaf = self.min_samples_leaf, max_features = self.max_features, **self.other_params)
        tree.fit(X_sample, y_sample)
        self.ensemble.append(tree)

        if self.oob_validation and X_oob is not None:
          y_oob_pred = tree.predict(X_oob)
          oob_score = accuracy_score(y_oob, y_oob_pred)
          self.oob_scores.append(oob_score)

      if self.oob_validation and self.oob_scores:
        self.average_oob_score = np.mean(self.oob_scores)

      return self


  def predict(self, X):

    if not self.ensemble:
      raise Exception("Not trained yet")

    n_samples = X.shape[0]
    unique_classes = np.unique(self.classes)
    n_classes = len(unique_classes)

    votes = np.zeros((n_samples, n_classes))

    for tree in self.ensemble:
      predictions = tree.predict(X)
      for i, res in enumerate(predictions):
        votes[i,res] +=1
    max_votes_list = np.argmax(votes, axis=1)

    return np.take(unique_classes, max_votes_list)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

X_train_forest = X1_train
X_test_forest = X1_test
y_train_forest = y1_train
y_test_forest = y1_test

sk_forest = RandomForestClassifier(
    n_estimators=100,
    max_features = 2,
    max_depth=None,
    bootstrap = True,
    min_samples_leaf=1
)

sk_tree = DecisionTreeClassifier
cart_tree = RSMWrapperCARTTree
c45_tree = RSMWrapperC45Tree

my_forest_sk_tree = MyRandomForestClassifier(
    sk_tree,
    n_estimators=20,
    max_features = 2,
    max_depth=None,
    bootstrap = True,
    min_samples_leaf=1
)

my_forest_cart_tree = MyRandomForestClassifier(
    cart_tree,
    n_estimators=20,
    max_features = 2,
    max_depth=None,
    bootstrap = True,
    min_samples_leaf=1
)

my_forest_c45_tree = MyRandomForestClassifier(
    c45_tree,
    n_estimators=20,
    max_features = 2,
    max_depth=None,
    bootstrap = True,
    min_samples_leaf=1,
    ccp_alpha = 0
)

In [ ]:
import numpy as np
import pandas as pd
import time
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
import asyncio
from joblib import Parallel, delayed, cpu_count
from tqdm import tqdm

"""
import sys
import os
import multiprocessing
print(f"multiprocessing.cpu_count(): {multiprocessing.cpu_count()}")
print(f"os.cpu_count(): {os.cpu_count()}")
print(f"joblib.cpu_count(): {cpu_count()}")
print(f"python: {sys.executable}")
print(f"hostname: {os.uname().nodename}")
sys.exit(1)
"""

# ================================
# 0. ПОДГОТОВКА ДАННЫХ
# ================================

# Предполагаем, что X_train_forest, X_test_forest, y_train_forest, y_test_forest уже загружены
# И модели уже созданы: sk_forest, my_forest_sk_tree, my_forest_cart_tree, my_forest_c45_tree

# ================================
# 1. ОБУЧЕНИЕ МОДЕЛЕЙ (СИНХРОННО)
# ================================

print("="*60)
print("ОБУЧЕНИЕ МОДЕЛЕЙ (СИНХРОННО)")
print("="*60)

models = {
    'Sklearn RF': sk_forest,
    'MyRF (sk_tree)': my_forest_sk_tree,
    #'MyRF (cart_tree)': my_forest_cart_tree,
    'MyRF (c45_tree)': my_forest_c45_tree
}

training_times = {}
async_training_times = {}
parallel_training_times = {}

for name, model in models.items():
    print(f"\n⏳ Обучение {name}...")
    start = time.time()

    if name == 'Sklearn RF':
        model.fit(X_train_forest, y_train_forest)
    else:
        model.fit(X_train_forest, y_train_forest)

    elapsed = time.time() - start
    training_times[name] = elapsed
    print(f"✅ {name} (синхронно) обучен за {elapsed:.4f} сек")

# ================================
# 1.2 АСИНХРОННОЕ ОБУЧЕНИЕ (для моих реализаций)
# ================================

print("\n" + "="*60)
print("АСИНХРОННОЕ ОБУЧЕНИЕ (await)")
print("="*60)

for name, model in models.items():
    if name != 'Sklearn RF':
        print(f"\n⏳ Асинхронное обучение {name}...")
        start = time.time()

        try:
            # Пытаемся использовать await (для Jupyter)
            await model._tree_tasks_completer(X_train_forest, y_train_forest)
        except RuntimeError:
            # Если не получилось - используем async_fit
            model.async_fit(X_train_forest, y_train_forest, lock=False)

        elapsed = time.time() - start
        async_training_times[name] = elapsed
        print(f"✅ {name} (асинхронно) обучен за {elapsed:.4f} сек")

# ================================
# 1.3 МУЛЬТИПРОЦЕССНОЕ ОБУЧЕНИЕ (parallel_fit) для моих реализаций
# ================================

print("\n" + "="*60)
print("МУЛЬТИПРОЦЕССНОЕ ОБУЧЕНИЕ (parallel_fit)")
print("="*60)

for name, model in models.items():
    if name != 'Sklearn RF':
        print(f"\n⏳ Мультипроцессное обучение {name}...")
        start = time.time()

        # ✅ Используем parallel_fit с n_jobs=-1 (все ядра)
        model.parallel_fit(X_train_forest, y_train_forest, n_jobs=-1)

        elapsed = time.time() - start
        parallel_training_times[name] = elapsed
        print(f"✅ {name} (мультипроцессно) обучен за {elapsed:.4f} сек")

# ================================
# 2. ПРЕДСКАЗАНИЯ И ACCURACY
# ================================

print("\n" + "="*60)
print("ПРЕДСКАЗАНИЯ И ACCURACY")
print("="*60)

results = []

for name, model in models.items():
    print(f"\n⏳ Предсказание {name}...")
    start = time.time()

    y_pred = model.predict(X_test_forest)

    elapsed = time.time() - start
    acc = accuracy_score(y_test_forest, y_pred)

    async_time = async_training_times.get(name, None)
    parallel_time = parallel_training_times.get(name, None)

    results.append({
        'Модель': name,
        'Accuracy': acc,
        'Время обучения (сек)': training_times[name],
        'Время async обучения (сек)': async_time,
        'Время parallel обучения (сек)': parallel_time,
        'Время предсказания (сек)': elapsed,
        'Количество деревьев': len(model.ensemble) if hasattr(model, 'ensemble') else len(model.estimators_),
        'Совпадение с sklearn': None
    })

    print(f"✅ {name}: Accuracy = {acc:.4f} (предсказание за {elapsed:.4f} сек)")

# ================================
# 3. СОВПАДЕНИЕ ПРЕДСКАЗАНИЙ
# ================================

print("\n" + "="*60)
print("СОВПАДЕНИЕ ПРЕДСКАЗАНИЙ СО SKLEARN")
print("="*60)

sk_pred = sk_forest.predict(X_test_forest)

for i, (name, model) in enumerate(models.items()):
    if name != 'Sklearn RF':
        y_pred = model.predict(X_test_forest)
        match = np.mean(sk_pred == y_pred)
        results[i]['Совпадение с sklearn'] = match
        print(f"✅ {name} совпадает со sklearn: {match:.4f}")

# ================================
# 4. СРАВНИТЕЛЬНАЯ ТАБЛИЦА
# ================================

print("\n" + "="*60)
print("СРАВНИТЕЛЬНАЯ ТАБЛИЦА")
print("="*60)

df_results = pd.DataFrame(results)
print(df_results.round(4))

# ================================
# 5. СРАВНЕНИЕ ВРЕМЕНИ (синхронный vs async vs parallel)
# ================================

print("\n" + "="*60)
print("СРАВНЕНИЕ ВРЕМЕНИ: СИНХРОННЫЙ vs ASYNC vs PARALLEL")
print("="*60)

comparison_data = []

for name in models.keys():
    if name != 'Sklearn RF':
        sync_time = training_times.get(name)
        async_time = async_training_times.get(name)
        parallel_time = parallel_training_times.get(name)

        if sync_time is not None and async_time is not None and parallel_time is not None:
            speedup_async = sync_time / async_time if async_time > 0 else 0
            speedup_parallel = sync_time / parallel_time if parallel_time > 0 else 0

            comparison_data.append({
                'Модель': name,
                'Синхронно (сек)': sync_time,
                'Async (сек)': async_time,
                'Parallel (сек)': parallel_time,
                'Ускорение async': f"{speedup_async:.2f}x",
                'Ускорение parallel': f"{speedup_parallel:.2f}x",
                'Лучший': 'Parallel' if speedup_parallel > speedup_async else 'Async' if speedup_async > 1 else 'Синхронный'
            })

            print(f"\n📊 {name}:")
            print(f"   Синхронно:      {sync_time:.4f} сек")
            print(f"   Асинхронно:     {async_time:.4f} сек (ускорение: {speedup_async:.2f}x)")
            print(f"   Мультипроцессно: {parallel_time:.4f} сек (ускорение: {speedup_parallel:.2f}x)")

            if speedup_parallel > speedup_async and speedup_parallel > 1:
                print(f"   🏆 Мультипроцессно БЫСТРЕЕ всех!")
            elif speedup_async > speedup_parallel and speedup_async > 1:
                print(f"   🏆 Асинхронно БЫСТРЕЕ всех!")
            elif speedup_parallel > 1 or speedup_async > 1:
                print(f"   ✅ Оба способа быстрее синхронного")
            else:
                print(f"   ⚠️ Асинхронный и мультипроцессный НЕ быстрее синхронного")

df_comparison = pd.DataFrame(comparison_data)
print("\n📊 Сводная таблица сравнения:")
print(df_comparison.round(4))

# ================================
# 6. ТЕСТ НА РАЗНЫХ n_jobs (для parallel_fit)
# ================================

print("\n" + "="*60)
print("ТЕСТ НА РАЗНЫХ n_jobs (parallel_fit)")
print("="*60)

n_jobs_list = [1, 2, 4, -1]  # -1 = все ядра
parallel_n_jobs_results = []

for name, model in models.items():
    if name != 'Sklearn RF':
        print(f"\n📊 {name} - тест n_jobs:")
        for n_jobs in n_jobs_list:
            # Создаем копию модели
            model_copy = MyRandomForestClassifier(
                model.tree_model,
                n_estimators=model.n_estimators,
                max_depth=model.max_depth,
                min_samples_leaf=model.min_samples_leaf,
                max_features=model.features_limit,
                oob_validation=model.oob_validation,
                bootstrap=model.bootstrap,
                **model.other_params
            )

            start = time.time()
            model_copy.parallel_fit(X_train_forest, y_train_forest, n_jobs=n_jobs)
            elapsed = time.time() - start

            acc = accuracy_score(y_test_forest, model_copy.predict(X_test_forest))

            parallel_n_jobs_results.append({
                'Модель': name,
                'n_jobs': n_jobs,
                'Время (сек)': elapsed,
                'Accuracy': acc
            })

            print(f"   n_jobs={n_jobs}: {elapsed:.4f} сек, Accuracy={acc:.4f}")

df_n_jobs = pd.DataFrame(parallel_n_jobs_results)
print("\n📊 Сравнение n_jobs:")
print(df_n_jobs.round(4))

# ================================
# 7. ИТОГОВОЕ ЗАКЛЮЧЕНИЕ
# ================================

print("\n" + "="*60)
print("ИТОГОВОЕ ЗАКЛЮЧЕНИЕ")
print("="*60)

best_acc = max(df_results['Accuracy'])
best_model = df_results[df_results['Accuracy'] == best_acc]['Модель'].values[0]

print(f"\n🏆 Лучшая модель по accuracy: {best_model} ({best_acc:.4f})")
print("\n📊 Все модели:")

for _, row in df_results.iterrows():
    status = "✅" if row['Accuracy'] >= 0.95 * best_acc else "⚠️"
    async_info = f" async: {row['Время async обучения (сек)']:.2f} сек" if pd.notna(row['Время async обучения (сек)']) else ""
    parallel_info = f" parallel: {row['Время parallel обучения (сек)']:.2f} сек" if pd.notna(row['Время parallel обучения (сек)']) else ""
    print(f"   {status} {row['Модель']}: {row['Accuracy']:.4f} (sync: {row['Время обучения (сек)']:.2f} сек{async_info}{parallel_info})")

# Проверка корректности
if all(row['Accuracy'] >= 0.90 * best_acc for _, row in df_results.iterrows()):
    print("\n✅ Все модели показывают хорошие результаты!")
else:
    print("\n⚠️ Некоторые модели показывают результаты хуже.")

if any(row['Совпадение с sklearn'] > 0.8 for _, row in df_results.iterrows()):
    print("✅ Некоторые модели хорошо совпадают со sklearn.")
else:
    print("⚠️ Ваши модели отличаются от sklearn.")

# Вывод для каждой модели
print("\n📈 Детальный вывод по моделям:")
for name in models.keys():
    if name == 'Sklearn RF':
        acc = df_results[df_results['Модель'] == name]['Accuracy'].values[0]
        print(f"\n📈 {name}: {acc:.4f}")
    else:
        acc = df_results[df_results['Модель'] == name]['Accuracy'].values[0]
        sync_time = training_times.get(name)
        async_time = async_training_times.get(name)
        parallel_time = parallel_training_times.get(name)
        print(f"\n📈 {name}: {acc:.4f} (sync: {sync_time:.2f} сек, async: {async_time:.2f} сек, parallel: {parallel_time:.2f} сек)")

print("\n✅ Тестирование завершено!")

ОБУЧЕНИЕ МОДЕЛЕЙ (СИНХРОННО)

⏳ Обучение Sklearn RF...
✅ Sklearn RF (синхронно) обучен за 0.4502 сек

⏳ Обучение MyRF (sk_tree)...


training process: 100%|██████████| 20/20 [00:00<00:00, 62.38it/s]


✅ MyRF (sk_tree) (синхронно) обучен за 0.3323 сек

⏳ Обучение MyRF (c45_tree)...


training process: 100%|██████████| 20/20 [00:19<00:00,  1.02it/s]


✅ MyRF (c45_tree) (синхронно) обучен за 19.6597 сек

АСИНХРОННОЕ ОБУЧЕНИЕ (await)

⏳ Асинхронное обучение MyRF (sk_tree)...
✅ MyRF (sk_tree) (асинхронно) обучен за 0.1027 сек

⏳ Асинхронное обучение MyRF (c45_tree)...
✅ MyRF (c45_tree) (асинхронно) обучен за 11.5991 сек

МУЛЬТИПРОЦЕССНОЕ ОБУЧЕНИЕ (parallel_fit)

⏳ Мультипроцессное обучение MyRF (sk_tree)...


parallel training process: 100%|██████████| 20/20 [00:03<00:00,  5.14it/s]


Average OOB Score: 0.7346
✅ MyRF (sk_tree) (мультипроцессно) обучен за 3.9212 сек

⏳ Мультипроцессное обучение MyRF (c45_tree)...


parallel training process:   0%|          | 0/20 [00:00<?, ?it/s]

KeyboardInterrupt: 

**Обертка над деревьями ITree**

In [ ]:
class ITreeWrapperCARTTree(CARTTree):
  pass
  #рандомно выбирать признак вместо rsm
  # новая функция для запоминания глубины
  # min_samples_leaf фиксирован как 1
  # убираем таргет и все подсчеты энтропии

In [ ]:
class ITreeWrapperC45Tree(C45Tree):
  pass

**Isolation Forest**

In [ ]:
class IsolationForest(MyRandomForestClassifier):
  pass
  # новый параметр размер выборки
  # новый параметр contamination=0.01 для отсечения норм от аномалий
  # подвыборка с возможностью бутстрапа
  # убираем таргет
  # средняя глубина